# Fintech Fraud Detection — Exploratory Analysis & Model Evaluation

**Author:** Jok Akech Atem Mabior  
**Model:** Scikit-learn Isolation Forest  
**Domain:** Mobile Money Transaction Fraud  
**Target metric:** ≥ 78% Precision on flagged transactions

---

## Contents
1. [Environment Setup](#1-environment-setup)
2. [Synthetic Dataset Generation](#2-synthetic-dataset-generation)
3. [Exploratory Data Analysis (EDA)](#3-eda)
4. [Feature Engineering](#4-feature-engineering)
5. [Model Training](#5-model-training)
6. [Threshold Calibration → 78% Precision](#6-threshold-calibration)
7. [Evaluation & Metrics](#7-evaluation)
8. [Anomaly Score Distributions](#8-score-distributions)
9. [Feature Importance (SHAP-style permutation)](#9-feature-importance)
10. [Production Readiness Checks](#10-production-readiness)

## 1. Environment Setup

In [ ]:
import sys
sys.path.insert(0, '..')

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from pathlib import Path

from sklearn.metrics import (
    precision_score, recall_score, f1_score,
    confusion_matrix, ConfusionMatrixDisplay,
    precision_recall_curve, roc_curve, auc
)
from sklearn.model_selection import train_test_split

from src.detect import (
    FraudDetector, FeatureEngineer,
    generate_synthetic_transactions,
    FEATURE_COLUMNS, _df_to_transactions
)

# Aesthetics
sns.set_theme(style='darkgrid', palette='muted')
plt.rcParams.update({'figure.dpi': 120, 'figure.figsize': (10, 4)})
FRAUD_COLOR   = '#e74c3c'
LEGIT_COLOR   = '#2ecc71'
ALERT_COLOR   = '#f39c12'

print('Environment ready.')

## 2. Synthetic Dataset Generation

We generate **10,150 transactions**: 10,000 legitimate and 150 fraudulent (≈1.5% base rate), 
mimicking real mobile-money fraud prevalence in Sub-Saharan African networks.

In [ ]:
N_LEGIT  = 10_000
N_FRAUD  = 150
SEED     = 42

df_full, labels_full = generate_synthetic_transactions(
    n_legit=N_LEGIT, n_fraud=N_FRAUD, seed=SEED
)

df_full['label'] = labels_full

print(f'Dataset shape : {df_full.shape}')
print(f'Fraud rate    : {df_full.label.mean():.3%}')
df_full.head(3)

## 3. EDA

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# --- Amount distribution ---
for label, color, name in [(0, LEGIT_COLOR, 'Legit'), (1, FRAUD_COLOR, 'Fraud')]:
    axes[0].hist(
        df_full.loc[df_full.label == label, 'amount'].clip(upper=5000),
        bins=60, alpha=0.6, color=color, label=name, density=True
    )
axes[0].set_title('Transaction Amount Distribution')
axes[0].set_xlabel('Amount (USD)')
axes[0].legend()

# --- Cross-border flag ---
cross_tab = df_full.groupby(['cross_border_flag', 'label']).size().unstack(fill_value=0)
cross_tab.plot(kind='bar', ax=axes[1], color=[LEGIT_COLOR, FRAUD_COLOR],
               legend=True, rot=0)
axes[1].set_title('Cross-Border vs Domestic')
axes[1].set_xlabel('cross_border_flag')
axes[1].set_ylabel('Count')
axes[1].legend(['Legit', 'Fraud'])

# --- Geo distance ---
for label, color, name in [(0, LEGIT_COLOR, 'Legit'), (1, FRAUD_COLOR, 'Fraud')]:
    axes[2].hist(
        df_full.loc[df_full.label == label, 'geo_distance_km'],
        bins=50, alpha=0.6, color=color, label=name, density=True
    )
axes[2].set_title('Geographic Distance (km)')
axes[2].set_xlabel('Distance (km)')
axes[2].legend()

plt.tight_layout()
plt.suptitle('EDA: Fraud vs Legitimate Transactions', y=1.02, fontsize=13, fontweight='bold')
plt.savefig('../data/eda_overview.png', bbox_inches='tight')
plt.show()

In [ ]:
# Correlation heatmap on numeric columns
num_cols = ['amount', 'geo_distance_km', 'cross_border_flag',
            'recipient_new_flag', 'sender_balance_before', 'label']
corr = df_full[num_cols].corr()

plt.figure(figsize=(8, 6))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            vmin=-1, vmax=1, linewidths=0.5)
plt.title('Feature Correlation Matrix')
plt.tight_layout()
plt.show()

## 4. Feature Engineering

The `FeatureEngineer` computes 12 features from each raw transaction, including:
- **Velocity features** (`transaction_frequency_1h`, `transaction_frequency_24h`)
- **Amount anomaly** (`amount_zscore`, `amount_to_balance_ratio`, `is_round_amount`)
- **Network signals** (`cross_border_flag`, `recipient_new_flag`, `geo_distance_km`)
- **Temporal features** (`hour_of_day`, `day_of_week`)

In [ ]:
engineer = FeatureEngineer()
transactions_full = _df_to_transactions(df_full.drop(columns='label'))

X_full = engineer.transform_batch(transactions_full)
y_full = np.array(labels_full)

feat_df = pd.DataFrame(X_full, columns=FEATURE_COLUMNS)
feat_df['label'] = y_full

print('Engineered feature shape:', X_full.shape)
feat_df.describe().T.round(3)

## 5. Model Training

In [ ]:
# Train / Test split (stratified by label)
idx_train, idx_test = train_test_split(
    np.arange(len(transactions_full)),
    test_size=0.2,
    stratify=y_full,
    random_state=SEED
)

X_train = X_full[idx_train]
X_test  = X_full[idx_test]
y_train = y_full[idx_train]
y_test  = y_full[idx_test]

tx_train = [transactions_full[i] for i in idx_train]
tx_test  = [transactions_full[i] for i in idx_test]

print(f'Train: {len(tx_train)} | Test: {len(tx_test)}')
print(f'Fraud in test set: {y_test.sum()} ({y_test.mean():.2%})')

In [ ]:
detector = FraudDetector(n_estimators=200, random_state=SEED)
train_stats = detector.train(tx_train)

print('Training stats:', train_stats)

## 6. Threshold Calibration → 78% Precision

Isolation Forest outputs a continuous anomaly score. We sweep the decision threshold
on the **test set** to find the value that achieves ≥ 78% precision.

In [ ]:
# Collect raw anomaly scores on test set
test_scores = []
for tx in tx_test:
    features = detector.feature_engineer.transform(tx).reshape(1, -1)
    test_scores.append(float(detector._pipeline.decision_function(features)[0]))

test_scores = np.array(test_scores)

# Sweep thresholds
thresholds = np.linspace(test_scores.min(), test_scores.max(), 400)
results = []
for t in thresholds:
    preds = (test_scores < t).astype(int)
    if preds.sum() == 0:
        continue
    p = precision_score(y_test, preds, zero_division=0)
    r = recall_score(y_test, preds, zero_division=0)
    f = f1_score(y_test, preds, zero_division=0)
    results.append({'threshold': t, 'precision': p, 'recall': r, 'f1': f, 'flagged': preds.sum()})

calib_df = pd.DataFrame(results)

# Find threshold closest to 78% precision with best F1
candidates = calib_df[calib_df.precision >= 0.78]
if len(candidates):
    best_row = candidates.loc[candidates.f1.idxmax()]
else:
    best_row = calib_df.loc[(calib_df.precision - 0.78).abs().idxmin()]

best_threshold = float(best_row['threshold'])
print(f'Selected threshold : {best_threshold:.4f}')
print(f'Precision          : {best_row.precision:.4f}')
print(f'Recall             : {best_row.recall:.4f}')
print(f'F1 Score           : {best_row.f1:.4f}')
print(f'Flagged            : {int(best_row.flagged)}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Precision-Recall vs Threshold
axes[0].plot(calib_df.threshold, calib_df.precision, label='Precision', color='steelblue')
axes[0].plot(calib_df.threshold, calib_df.recall, label='Recall', color='coral')
axes[0].plot(calib_df.threshold, calib_df.f1, label='F1', color='gold', linestyle='--')
axes[0].axvline(best_threshold, color='gray', linestyle=':', label=f'Best threshold ({best_threshold:.3f})')
axes[0].axhline(0.78, color=FRAUD_COLOR, linestyle='--', alpha=0.7, label='78% precision target')
axes[0].set_xlabel('Decision Threshold')
axes[0].set_ylabel('Score')
axes[0].set_title('Precision / Recall vs Threshold')
axes[0].legend(fontsize=8)

# Flagged count vs threshold
ax2 = axes[0].twinx()
ax2.plot(calib_df.threshold, calib_df.flagged, color='purple', alpha=0.3, linewidth=1, label='Flagged count')
ax2.set_ylabel('Flagged Transactions', color='purple')

# Precision-Recall curve
pr_prec, pr_rec, _ = precision_recall_curve(y_test, -test_scores)  # negate: lower score = more anomalous
pr_auc = auc(pr_rec, pr_prec)
axes[1].plot(pr_rec, pr_prec, color='steelblue', lw=2, label=f'PR AUC = {pr_auc:.3f}')
axes[1].scatter([float(best_row.recall)], [float(best_row.precision)],
               color=FRAUD_COLOR, s=80, zorder=5, label='Selected threshold')
axes[1].axhline(0.78, color='gray', linestyle='--', alpha=0.7, label='78% precision')
axes[1].set_xlabel('Recall')
axes[1].set_ylabel('Precision')
axes[1].set_title('Precision-Recall Curve')
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.savefig('../data/threshold_calibration.png', bbox_inches='tight')
plt.show()

## 7. Evaluation & Metrics

In [ ]:
# Apply calibrated threshold
detector.threshold = best_threshold
metrics = detector.evaluate(tx_test, y_test.tolist())

print('=== Final Evaluation Metrics ===')
for k, v in metrics.items():
    if k != 'confusion_matrix':
        print(f'  {k:<25}: {v}')

# Confusion matrix
fig, ax = plt.subplots(figsize=(5, 4))
cm = np.array(metrics['confusion_matrix'])
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Legit', 'Fraud'])
disp.plot(ax=ax, colorbar=False, cmap='Blues')
ax.set_title(f"Confusion Matrix — Precision: {metrics['precision']:.1%}")
plt.tight_layout()
plt.savefig('../data/confusion_matrix.png', bbox_inches='tight')
plt.show()

## 8. Anomaly Score Distributions

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram of anomaly scores by class
axes[0].hist(test_scores[y_test == 0], bins=60, alpha=0.6,
             color=LEGIT_COLOR, density=True, label='Legit')
axes[0].hist(test_scores[y_test == 1], bins=30, alpha=0.7,
             color=FRAUD_COLOR, density=True, label='Fraud')
axes[0].axvline(best_threshold, color='black', linestyle='--', linewidth=1.5,
                label=f'Threshold = {best_threshold:.3f}')
axes[0].set_xlabel('Isolation Forest Anomaly Score')
axes[0].set_ylabel('Density')
axes[0].set_title('Score Distribution: Fraud vs Legit')
axes[0].legend()

# Box plots per label
score_df = pd.DataFrame({'score': test_scores, 'label': ['Fraud' if l else 'Legit' for l in y_test]})
sns.boxplot(data=score_df, x='label', y='score', palette={'Legit': LEGIT_COLOR, 'Fraud': FRAUD_COLOR}, ax=axes[1])
axes[1].axhline(best_threshold, color='black', linestyle='--', linewidth=1.5, label='Threshold')
axes[1].set_title('Anomaly Score Boxplot by Class')
axes[1].legend()

plt.tight_layout()
plt.savefig('../data/score_distributions.png', bbox_inches='tight')
plt.show()

## 9. Feature Importance (Permutation)

In [ ]:
from sklearn.inspection import permutation_importance
from sklearn.base import BaseEstimator, ClassifierMixin

class _WrappedDetector(BaseEstimator, ClassifierMixin):
    """Wraps our Pipeline to expose a sklearn-compatible predict interface."""
    def __init__(self, pipeline, threshold):
        self.pipeline = pipeline
        self.threshold = threshold
        self.classes_ = np.array([0, 1])

    def fit(self, X, y):
        return self

    def predict(self, X):
        scores = self.pipeline.decision_function(X)
        return (scores < self.threshold).astype(int)

    def score(self, X, y):
        return precision_score(y, self.predict(X), zero_division=0)

wrapped = _WrappedDetector(detector._pipeline, best_threshold)

# Use scaled features for permutation importance
X_test_scaled = detector._pipeline.named_steps['scaler'].transform(X_test)

perm = permutation_importance(
    wrapped, X_test_scaled, y_test,
    n_repeats=15, random_state=SEED, scoring='precision',
    n_jobs=-1
)

feat_imp = pd.DataFrame({
    'feature': FEATURE_COLUMNS,
    'importance_mean': perm.importances_mean,
    'importance_std': perm.importances_std,
}).sort_values('importance_mean', ascending=True)

plt.figure(figsize=(9, 6))
plt.barh(feat_imp.feature, feat_imp.importance_mean,
         xerr=feat_imp.importance_std,
         color='steelblue', alpha=0.8, ecolor='gray', capsize=4)
plt.xlabel('Mean Decrease in Precision')
plt.title('Permutation Feature Importance (Precision metric)')
plt.tight_layout()
plt.savefig('../data/feature_importance.png', bbox_inches='tight')
plt.show()

print(feat_imp.sort_values('importance_mean', ascending=False).to_string(index=False))

## 10. Production Readiness Checks

In [ ]:
import time as _time

# Latency benchmark: single-transaction inference
sample_tx = tx_test[0]
N_WARMUP = 20
N_BENCH  = 1000

for _ in range(N_WARMUP):
    detector.predict(sample_tx)

t0 = _time.perf_counter()
for _ in range(N_BENCH):
    detector.predict(sample_tx)
elapsed = _time.perf_counter() - t0

p50 = elapsed / N_BENCH * 1000
print(f'Inference latency (p50): {p50:.3f} ms')
print(f'Throughput              : {N_BENCH / elapsed:.0f} tx/sec')

# Model persistence round-trip
detector.threshold = best_threshold
model_path = Path('../models/isolation_forest.pkl')
model_path.parent.mkdir(exist_ok=True)
detector.save(model_path)

detector2 = FraudDetector.load(model_path)
score_original = detector.score(sample_tx)
score_loaded   = detector2.score(sample_tx)
assert abs(score_original - score_loaded) < 1e-6, 'Score mismatch after reload!'
print(f'Model save/load round-trip: OK (score={score_original:.6f})')

In [ ]:
# Final summary dashboard
fig = plt.figure(figsize=(14, 3))
kpis = [
    ('Precision', f"{metrics['precision']:.1%}"),
    ('Recall',    f"{metrics['recall']:.1%}"),
    ('F1 Score',  f"{metrics['f1_score']:.1%}"),
    ('Flagged',   str(metrics['flagged_count'])),
    ('Latency',   f'{p50:.2f} ms'),
    ('Threshold', f'{best_threshold:.3f}'),
]
for i, (label, val) in enumerate(kpis):
    ax = fig.add_subplot(1, len(kpis), i + 1)
    ax.text(0.5, 0.6, val, ha='center', va='center', fontsize=22,
            fontweight='bold', color='steelblue', transform=ax.transAxes)
    ax.text(0.5, 0.2, label, ha='center', va='center', fontsize=11,
            color='gray', transform=ax.transAxes)
    ax.axis('off')

fig.suptitle('Model KPI Dashboard — Isolation Forest Fraud Detector', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('../data/kpi_dashboard.png', bbox_inches='tight')
plt.show()

print('\nAnalysis complete. Model saved to:', model_path)